# Readmission Risk — Data Preprocessing

Prepares the Diabetes 130-US Hospitals dataset for modeling: predicting whether
a patient will be **readmitted within 30 days** of discharge.

**Prerequisite:** run `src/download_data.py` first so `data/diabetic_data.csv` exists.


In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/diabetic_data.csv")
print(f"Loaded: {df.shape}")
df.head()

## 1. Deduplicate to first encounter per patient

The dataset contains repeat visits for the same patient. Using every encounter as an
independent row would leak information (the model could learn from a patient's *later*
visits when predicting an *earlier* one). We keep only each patient's first encounter.

In [ ]:
df = df.sort_values("encounter_id").drop_duplicates(subset="patient_nbr", keep="first")
print(f"After dedup: {df.shape}")

## 2. Drop unusable columns

- `weight` is 97% missing — unusable
- `payer_code` is administrative, not clinically predictive, and 40% missing
- `encounter_id` / `patient_nbr` are identifiers, not features
- A handful of rows have invalid `gender` values — drop them

In [ ]:
df = df.drop(columns=["weight", "payer_code", "encounter_id", "patient_nbr"])
df = df[df["gender"] != "Unknown/Invalid"]
df = df.replace("?", np.nan)
print(f"Shape: {df.shape}")

## 3. Define the target: 30-day readmission

The raw `readmitted` column has three values: `NO`, `>30`, `<30`. Clinically and for the
Hospital Readmissions Reduction Program, the meaningful outcome is readmission **within
30 days**, so we binarize to that.

In [ ]:
df["readmit_30d"] = (df["readmitted"] == "<30").astype(int)
df = df.drop(columns=["readmitted"])
print(df["readmit_30d"].value_counts(normalize=True))

## 4. Group diagnosis codes into clinical categories

`diag_1/2/3` are raw ICD-9 codes with hundreds of unique values — too sparse to use directly.
We group them into standard clinical categories (circulatory, respiratory, digestive,
diabetes, injury, musculoskeletal, genitourinary, neoplasms, other), following the grouping
used in the original UCI dataset research.

In [ ]:
def group_diagnosis(code):
    if pd.isna(code):
        return "Missing"
    code = str(code)
    if code.startswith("V") or code.startswith("E"):
        return "Other"
    try:
        code_num = float(code)
    except ValueError:
        return "Other"
    if 390 <= code_num <= 459 or code_num == 785:
        return "Circulatory"
    elif 460 <= code_num <= 519 or code_num == 786:
        return "Respiratory"
    elif 520 <= code_num <= 579 or code_num == 787:
        return "Digestive"
    elif code_num == 250:
        return "Diabetes"
    elif 800 <= code_num <= 999:
        return "Injury"
    elif 710 <= code_num <= 739:
        return "Musculoskeletal"
    elif 580 <= code_num <= 629 or code_num == 788:
        return "Genitourinary"
    elif 140 <= code_num <= 239:
        return "Neoplasms"
    else:
        return "Other"

for col in ["diag_1", "diag_2", "diag_3"]:
    df[col + "_group"] = df[col].apply(group_diagnosis)
df = df.drop(columns=["diag_1", "diag_2", "diag_3"])

df["diag_1_group"].value_counts()

## 5. Fill remaining missing values

Remaining categorical missingness (e.g. `race`, `medical_specialty`) is filled with its
own `"Missing"` category rather than dropped or imputed — missingness can itself carry
signal (e.g. a missing specialty may correlate with which department admitted the patient).

In [ ]:
cat_cols = df.select_dtypes(include="object").columns
for col in cat_cols:
    df[col] = df[col].fillna("Missing")

print(f"Final shape: {df.shape}")
print(f"Remaining nulls: {df.isnull().sum().sum()}")

## 6. Save processed dataset

In [ ]:
df.to_csv("data/processed_diabetes.csv", index=False)
print("Saved data/processed_diabetes.csv")
print(f"\nColumns ({df.shape[1]}):")
print(list(df.columns))